# Flood Impact Text — TF-IDF Analysis

Explore the free-text impact descriptions in `flood_stages.parquet` across the four severity levels: action, flood, moderate, major.

(Term Frequency-Inverse Document Frequency) is a numerical statistic used to reflect how important a word is to a document within a collection or corpus

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import rplot

plt.style.use('ryan')

DATA_PATH = '/home/ryan/data/flood_hazard/metadata/flood_stages.parquet'
LEVELS = ['action', 'flood', 'moderate', 'major']
IMPACT_COLS = [f'{lvl}_impact' for lvl in LEVELS]

## 1. Load & overview

In [ ]:
df = pd.read_parquet(DATA_PATH)
print(f'Total sites: {len(df):,}')
print()
for col in IMPACT_COLS:
    texts = df[col].dropna()
    lengths = texts.str.split().str.len()
    print(f'{col}: {len(texts):,} non-null | median {lengths.median():.0f} words | max {lengths.max()} words')

In [ ]:
# Word-count distributions per level
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=False)
for ax, col, lvl in zip(axes, IMPACT_COLS, LEVELS):
    lengths = df[col].dropna().str.split().str.len()
    ax.hist(lengths, bins=40, color='steelblue', edgecolor='none')
    ax.set_title(lvl.capitalize())
    ax.set_xlabel('Word count')
    ax.set_ylabel('Sites' if ax == axes[0] else '')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(5))
plt.suptitle('Description length by severity level', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 2. Preprocessing

In [ ]:
# Custom stopwords — standard English list plus domain-specific noise
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

EXTRA_STOPS = {
    # generic flood/water words present at every level
    'flood', 'flooding', 'floodwaters', 'water', 'waters', 'river', 'stream',
    'creek', 'stage', 'flow', 'level', 'levels',
    # severity-level labels that contaminate cross-level comparison
    # (descriptions typically open with "Minor flooding...", "Moderate flooding...", etc.)
    'minor', 'moderate', 'major', 'action', 'significant', 'severe',
    'begins', 'begin', 'occur', 'occurs', 'possible', 'expected', 'progress',
    # directional / geographic filler
    'near', 'along', 'upstream', 'downstream', 'area', 'areas',
    # numeric unit tokens
    'ft', 'cfs', 'feet',
}
STOP_WORDS = ENGLISH_STOP_WORDS.union(EXTRA_STOPS)

def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)   # strip punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()
    return text

texts_by_level = {}
for col, lvl in zip(IMPACT_COLS, LEVELS):
    texts_by_level[lvl] = df[col].dropna().map(clean).tolist()

for lvl, texts in texts_by_level.items():
    print(f'{lvl}: {len(texts):,} descriptions')

## 3. Within-level TF-IDF — top terms per severity

Each description is a document. TF-IDF scores tell us which terms are important in a given description *relative to how common they are across all descriptions at that level*. We take the mean score across all documents to surface terms that are broadly representative.

In [ ]:
TOP_N = 20

within_level_top = {}
for lvl, texts in texts_by_level.items():
    vec = TfidfVectorizer(
        stop_words=list(STOP_WORDS),
        ngram_range=(1, 2),
        min_df=5,           # must appear in at least 5 descriptions
        max_df=0.9,         # drop near-universal terms
        sublinear_tf=True,
    )
    mat = vec.fit_transform(texts)
    mean_scores = np.asarray(mat.mean(axis=0)).flatten()
    terms = vec.get_feature_names_out()
    top_idx = mean_scores.argsort()[::-1][:TOP_N]
    within_level_top[lvl] = pd.Series(mean_scores[top_idx], index=terms[top_idx])

# Display as table
compare = pd.DataFrame(within_level_top)
compare.index.name = 'term'
compare

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 6), sharey=False)
colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']

for ax, lvl, color in zip(axes, LEVELS, colors):
    s = within_level_top[lvl].sort_values()
    ax.barh(s.index, s.values, color=color, edgecolor='none')
    ax.set_title(lvl.capitalize(), fontsize=12)
    ax.set_xlabel('Mean TF-IDF')
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Top terms per severity level (within-level TF-IDF)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. Cross-level TF-IDF — what's *distinctive* to each level?

Concatenate all descriptions at each level into one mega-document, then run TF-IDF across the four mega-documents. High scores = terms that appear frequently at this level but not at others.

In [ ]:
corpus = [' '.join(texts_by_level[lvl]) for lvl in LEVELS]

vec_cross = TfidfVectorizer(
    stop_words=list(STOP_WORDS),
    ngram_range=(1, 2),
    sublinear_tf=True,
)
mat_cross = vec_cross.fit_transform(corpus)
terms_cross = vec_cross.get_feature_names_out()

cross_df = pd.DataFrame(
    mat_cross.toarray(),
    index=LEVELS,
    columns=terms_cross,
).T

# Top N distinctive terms per level
TOP_CROSS = 25
cross_top = {lvl: cross_df[lvl].nlargest(TOP_CROSS) for lvl in LEVELS}

fig, axes = plt.subplots(1, 4, figsize=(18, 7), sharey=False)
for ax, lvl, color in zip(axes, LEVELS, colors):
    s = cross_top[lvl].sort_values()
    ax.barh(s.index, s.values, color=color, edgecolor='none')
    ax.set_title(f'{lvl.capitalize()} (distinctive)', fontsize=12)
    ax.set_xlabel('TF-IDF score')
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Terms most distinctive to each severity level (cross-level TF-IDF)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. Term overlap across levels

Heatmap of how the top terms at any level score across all levels — reveals which terms are level-specific vs. shared.

In [ ]:
# Union of top-N distinctive terms across all levels
all_top_terms = pd.Index([])
for lvl in LEVELS:
    all_top_terms = all_top_terms.union(cross_top[lvl].index)

heatmap_df = cross_df.loc[all_top_terms].copy()
# Normalise each row so colours reflect relative distinctiveness
heatmap_df = heatmap_df.div(heatmap_df.max(axis=1), axis=0)
# Sort rows by which level they peak in, then by score
heatmap_df['peak_level'] = heatmap_df.idxmax(axis=1)
heatmap_df['peak_score'] = heatmap_df[LEVELS].max(axis=1)
heatmap_df = heatmap_df.sort_values(['peak_level', 'peak_score'], ascending=[True, False])
heatmap_df = heatmap_df[LEVELS]

fig, ax = plt.subplots(figsize=(6, max(8, len(heatmap_df) * 0.22)))
im = ax.imshow(heatmap_df.values, aspect='auto', cmap=rplot.SEQ_HALINE.reversed(), vmin=0, vmax=1)
ax.set_xticks(range(4))
ax.set_xticklabels([l.capitalize() for l in LEVELS], fontsize=11)
ax.set_yticks(range(len(heatmap_df)))
ax.set_yticklabels(heatmap_df.index, fontsize=7)
plt.colorbar(im, ax=ax, label='Relative TF-IDF (row-normalised)', shrink=0.5, aspect=30)
ax.set_title('Term distinctiveness across severity levels', fontsize=12)
ax.grid(False)
plt.tight_layout()
plt.show()

## 6. Bigram spotlight

Bigrams often capture impact language better than unigrams (e.g. *road closed*, *homes flooded*). Pull the top bigrams per level from the cross-level results.

In [ ]:
print('Top bigrams per level (cross-level TF-IDF)\n')
for lvl in LEVELS:
    bigrams = cross_top[lvl][cross_top[lvl].index.str.contains(' ')].head(10)
    print(f'--- {lvl.upper()} ---')
    for term, score in bigrams.items():
        print(f'  {term:<35} {score:.4f}')
    print()